# 03 – Transformer stress (Liander "Netbewust publiek laden – knelpunt")

**Goal:** turn the transformer peak-load data into a grid-stress measure per buurt.
**Input:** `1. Netbewust publiek laden Liander - Dataset knelpunt 2026-01.csv` (raw CSV; the Excel version lost its decimal commas)
**Outputs:** `Data/processed/transformers_buurt2023.csv`, `Data/processed/buurt_transformer_stress.csv`

## Data
- 31,367 transformers, one row each. Peak load, capacity and NBL limit, all in kW.
- NBL = *netbewust laden* limit: the level above which public EV charging is limited → early-warning threshold, slightly below capacity.
- Snapshot: peaks measured between Sep 2024 and Jul 2025 (not a time series).

## Cleaning
- Only 0.4 kV (neighbourhood) transformers kept: 31,323 of 31,367.
- Removed: 4 with capacity/NBL = 0 (impossible), 15 with peak = 0 (no measurement) → 31,304 kept.
- Utilisation = peak / capacity. Extremes: 5 transformers > 200% (e.g. 272 kW on 50 kW → capacity record likely wrong) → `utilisation_capped` clipped at 2.0; original kept.

## Key observation (hypothesis, tested in notebook 04)
- ~98% of peaks occur in April–July. First interpretation: solar feed-in drives stress.
- **Tested in 04: not supported per home.** Stressed buurten have *lower* solar shares. Stress is linked to load relative to capacity (connections and kWh per transformer). Solar feed-in per kW capacity raises average utilisation but not the NBL stress rate.
- **Tested again in 08 (Liander PV):** summer peaks (~98%) occur equally in low- and high-solar buurten → the summer timing is likely a measurement artefact (e.g. most peaks measured in spring/summer 2025), not proof of feed-in. Stress = mix of consumption and feed-in. To verify with Liander/Deloitte.

## Linking to buurten
- Spatial join of transformer coordinates to the 2023 buurt shapes (31,285 within, 19 nearest ≤ 500 m).
- Only 67% match Liander's reported buurt code: 9% of reported codes do not exist in 2023 (different year, e.g. merged municipalities), the rest are mostly neighbouring buurten.
- **Decision:** use our own 2023 link (consistent with the consumption data); reported code kept for transparency.

## Buurt level (5,523 buurten)
- `stress_nbl` = at least one transformer above the NBL limit → 10.1% of buurten (**main target**).
- `stress_capacity` = at least one transformer above capacity → 6.6% (stricter check).
- Bias: buurten with more transformers have more chance of "at least one" → `n_transformers` used as a feature; `util_weighted` as a size-independent measure.
- Classes are imbalanced (10% stressed). Not corrected with class weights: the model is judged on ranking (ROC-AUC / PR-AUC), which does not need it (see notebook 08).

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../Data/raw")
PROCESSED = Path("../Data/processed")
TRAFO = RAW / "4 Liander" / "Raw Data" / "1. Netbewust publiek laden Liander - Dataset knelpunt 2026-01.csv"

# look at the raw text first
with open(TRAFO, encoding="utf-8-sig", errors="replace") as f:
    print("HEADER:", f.readline()[:200])
    print("ROW   :", f.readline()[:200])

# read everything as text; pandas detects the separator
tr = pd.read_csv(TRAFO, sep=None, engine="python", dtype=str, encoding="utf-8-sig")
tr.columns = tr.columns.str.strip()
print("\nShape:", tr.shape)
print("Unique transformers:", tr["CONDUCTINGEQUIPMENT_NAME"].nunique())

# what kinds of values are in the key columns?
for col in ["MKTPSRTYPE_PSRTYPE", "BASEVOLTAGE_NOMINALVOLTAGE", "ANALOG_NAME",
            "ANALOG_UNITMULTIPLIER", "ANALOG_UNITSYMBOL", "OPERATIONALLIMITSET_NAME",
            "ACTIVEPOWERLIMIT_UNITMULTIPLIER", "OPERATIONALLIMITSET_NAME_NBL"]:
    print("\n---", col)
    print(tr[col].value_counts(dropna=False).head(6))

HEADER: SUBSTATION_NAME;CONDUCTINGEQUIPMENT_NAME;MKTPSRTYPE_PSRTYPE;NOMINALVOLTAGE_UNITMULTIPLIER;NOMINALVOLTAGE_UNITSYMBOL;BASEVOLTAGE_NOMINALVOLTAGE;STREETADDRESS_POSTALCODE;STREETDETAIL_NAME;STREETDETAIL_N
ROW   : 9 009 404;9009404-EL-T01;Transformator;k;V;0.4;8312AM;TOERNOOILAAN;4;BU01711045;NOORDOOSTPOLDER;CREIL;FLEVOLAND;EPSG:4326;52,76301542;5,654751933;Piek Belasting;ActivePower;k;W;37;18-6-2025;Capaciteit

Shape: (31367, 30)
Unique transformers: 31367

--- MKTPSRTYPE_PSRTYPE
MKTPSRTYPE_PSRTYPE
Transformator    31367
Name: count, dtype: int64

--- BASEVOLTAGE_NOMINALVOLTAGE
BASEVOLTAGE_NOMINALVOLTAGE
0.4     31323
3.0        21
0.6        12
10.0        7
20.0        2
\\N         1
Name: count, dtype: int64

--- ANALOG_NAME
ANALOG_NAME
Piek Belasting    31367
Name: count, dtype: int64

--- ANALOG_UNITMULTIPLIER
ANALOG_UNITMULTIPLIER
k    31367
Name: count, dtype: int64

--- ANALOG_UNITSYMBOL
ANALOG_UNITSYMBOL
W    31367
Name: count, dtype: int64

--- OPERATIONALLIMITSET_NAME
O

In [2]:
def to_num(s):
    return pd.to_numeric(s.str.strip().str.replace(",", ".", regex=False), errors="coerce")

t = pd.DataFrame({
    "transformer_id":     tr["CONDUCTINGEQUIPMENT_NAME"],
    "voltage_kv":         to_num(tr["BASEVOLTAGE_NOMINALVOLTAGE"]),
    "postcode":           tr["STREETADDRESS_POSTALCODE"].str.strip(),
    "buurtcode_reported": tr["STREETDETAIL_CODE"].str.strip(),
    "municipality":       tr["TOWNDETAIL_NAME"].str.strip(),
    "lat":                to_num(tr["POSITIONPOINT_YPOSITION"]),
    "lon":                to_num(tr["POSITIONPOINT_XPOSITION"]),
    "peak_kw":            to_num(tr["ANALOGVALUE_VALUE"]),
    "peak_date":          pd.to_datetime(tr["ANALOGVALUE_TIMESTAMP"].str.strip(), dayfirst=True, errors="coerce"),
    "capacity_kw":        to_num(tr["ACTIVEPOWERLIMIT_VALUE"]),
    "nbl_limit_kw":       to_num(tr["ACTIVEPOWERLIMIT_VALUE_NBL"]),
})

# keep only the 0.4 kV neighbourhood transformers
t = t[t["voltage_kv"] == 0.4].copy()
print("0.4 kV transformers:", len(t))

print("\nMissing values:")
print(t.isna().sum())

print("\nNumbers:")
print(t[["peak_kw", "capacity_kw", "nbl_limit_kw", "lat", "lon"]].describe().round(2))

print("\nPeak dates from", t["peak_date"].min().date(), "to", t["peak_date"].max().date())

print("\nMost common capacities (kW):")
print(t["capacity_kw"].value_counts().head(10))

0.4 kV transformers: 31323

Missing values:
transformer_id        0
voltage_kv            0
postcode              0
buurtcode_reported    0
municipality          0
lat                   0
lon                   0
peak_kw               0
peak_date             0
capacity_kw           0
nbl_limit_kw          0
dtype: int64

Numbers:
        peak_kw  capacity_kw  nbl_limit_kw       lat       lon
count  31323.00     31323.00      31323.00  31323.00  31323.00
mean     147.42       363.53        335.64     52.39      5.38
std      107.59       184.98        170.42      0.40      0.59
min        0.00         0.00          0.00     51.74      4.34
25%       61.00       250.00        220.00     52.11      4.84
50%      131.00       400.00        352.00     52.33      5.45
75%      208.00       400.00        384.00     52.64      5.88
max      864.00      6000.00       5280.00     53.49      6.79

Peak dates from 2024-09-03 to 2025-07-10

Most common capacities (kW):
capacity_kw
400     10385
630 

In [3]:
# 1. peak date period
print("Peak dates:", t["peak_date"].min().date(), "to", t["peak_date"].max().date())
print("Peaks per month:")
print(t["peak_date"].dt.month.value_counts().sort_index().to_string())

# 2. problem values
print("\nCapacity = 0:     ", (t["capacity_kw"] == 0).sum())
print("NBL limit = 0:    ", (t["nbl_limit_kw"] == 0).sum())
print("Peak = 0:         ", (t["peak_kw"] == 0).sum())
print("NBL > capacity:   ", (t["nbl_limit_kw"] > t["capacity_kw"]).sum())

# 3. remove impossible rows (no capacity), then compute utilisation
t = t[(t["capacity_kw"] > 0) & (t["nbl_limit_kw"] > 0)].copy()
t["utilisation"] = t["peak_kw"] / t["capacity_kw"]
t["above_nbl"] = t["peak_kw"] > t["nbl_limit_kw"]
t["above_capacity"] = t["peak_kw"] > t["capacity_kw"]

print("\nTransformers kept:", len(t))
print("\nUtilisation (peak / capacity):")
print(t["utilisation"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).round(2).to_string())
print(f"\nAbove NBL limit: {t['above_nbl'].sum():,} ({100 * t['above_nbl'].mean():.1f}%)")
print(f"Above capacity:  {t['above_capacity'].sum():,} ({100 * t['above_capacity'].mean():.1f}%)")

Peak dates: 2024-09-03 to 2025-07-10
Peaks per month:
peak_date
2        25
3       104
4      8336
5      2017
6     16582
7      3975
9        11
10       25
11      239
12        9

Capacity = 0:      4
NBL limit = 0:     4
Peak = 0:          15
NBL > capacity:    0

Transformers kept: 31319

Utilisation (peak / capacity):
count    31319.00
mean         0.42
std          0.24
min          0.00
50%          0.40
75%          0.57
90%          0.73
95%          0.82
99%          1.03
max          5.44

Above NBL limit: 651 (2.1%)
Above capacity:  407 (1.3%)


In [4]:
# remove transformers without a measured peak
t = t[t["peak_kw"] > 0].copy()
print("Transformers kept:", len(t))

# how many transformers at each utilisation level?
bins = [0, 0.5, 0.8, 1.0, 1.5, 2.0, 3.0, 10]
print("\nUtilisation groups:")
print(pd.cut(t["utilisation"], bins=bins).value_counts().sort_index().to_string())

# the 15 most extreme transformers
print("\nMost extreme transformers:")
print(t.sort_values("utilisation", ascending=False)
       [["transformer_id", "municipality", "peak_kw", "capacity_kw", "nbl_limit_kw", "utilisation", "peak_date"]]
       .head(15).to_string(index=False))

Transformers kept: 31304

Utilisation groups:
utilisation
(0.0, 0.5]     20402
(0.5, 0.8]      9022
(0.8, 1.0]      1473
(1.0, 1.5]       391
(1.5, 2.0]        11
(2.0, 3.0]         4
(3.0, 10.0]        1

Most extreme transformers:
transformer_id        municipality  peak_kw  capacity_kw  nbl_limit_kw  utilisation  peak_date
3009419-EL-T01           AMSTERDAM      272           50            48       5.4400 2025-04-18
6004326-EL-T01                TIEL      695          250           240       2.7800 2025-06-24
1003203-EL-T01          LEEUWARDEN      250          100            96       2.5000 2025-06-13
6002908-EL-T01            NIJMEGEN      378          160           153       2.3625 2025-06-23
3016765-EL-T01           AMSTERDAM      808          400           384       2.0200 2025-07-10
9008837-EL-T01          LEEUWARDEN      469          250           240       1.8760 2025-06-12
3003063-EL-T01           AMSTERDAM      697          400           384       1.7425 2025-05-06
6004327

In [5]:
import geopandas as gpd

# 1. cap utilisation at 200%
t["utilisation_capped"] = t["utilisation"].clip(upper=2.0)

# 2. load the 2023 buurt shapes (land only), same as notebook 02
buurten = gpd.read_file(RAW / "External" / "cbs_buurten_2023_simplified.geojson")
land = buurten[(buurten["water"] != "JA") & (buurten["gemeentenaam"] != "Buitenland")]
land = land[["buurtcode", "buurtnaam", "gemeentenaam", "geometry"]]

# 3. transformer points -> buurt (within)
pts = gpd.GeoDataFrame(t, geometry=gpd.points_from_xy(t["lon"], t["lat"]), crs="EPSG:4326")
j = gpd.sjoin(pts, land, how="left", predicate="within").drop_duplicates("transformer_id")
j = j.drop(columns="index_right")
j["method"] = np.where(j["buurtcode"].notna(), "within", None)

# 4. unmatched -> nearest buurt within 500 m
miss = j[j["buurtcode"].isna()].drop(columns=["buurtcode", "buurtnaam", "gemeentenaam"])
near = gpd.sjoin_nearest(miss.to_crs("EPSG:28992"), land.to_crs("EPSG:28992"),
                         how="left", max_distance=500).drop_duplicates("transformer_id")
near = near.drop(columns="index_right")
near["method"] = np.where(near["buurtcode"].notna(), "nearest", None)
j = pd.concat([j[j["buurtcode"].notna()], near.to_crs("EPSG:4326")], ignore_index=True)

print("Transformers:", len(j))
print(j["method"].value_counts(dropna=False).to_string())
same = (j["buurtcode"] == j["buurtcode_reported"]).mean()
print(f"\nOur buurt = reported buurt: {100 * same:.1f}%")

Transformers: 31304
method
within     31285
nearest       19

Our buurt = reported buurt: 67.0%


In [6]:
valid_2023 = set(land["buurtcode"])
j["reported_exists_2023"] = j["buurtcode_reported"].isin(valid_2023)
j["codes_differ"] = j["buurtcode"] != j["buurtcode_reported"]

print("Rows: codes differ (True/False) x reported code exists in 2023 (True/False)")
print(pd.crosstab(j["codes_differ"], j["reported_exists_2023"], margins=True))

# among differing codes that DO exist in 2023: how close are they?
d = j[j["codes_differ"] & j["reported_exists_2023"]]
print(f"\nDiffering but existing codes: {len(d)}")
print(f"  same municipality: {100 * (d['buurtcode'].str[:6] == d['buurtcode_reported'].str[:6]).mean():.1f}%")
print(f"  same wijk:         {100 * (d['buurtcode'].str[:8] == d['buurtcode_reported'].str[:8]).mean():.1f}%")

print("\nMunicipalities with the most differences:")
print(j.loc[j["codes_differ"], "municipality"].value_counts().head(10).to_string())

Rows: codes differ (True/False) x reported code exists in 2023 (True/False)
reported_exists_2023  False   True    All
codes_differ                             
False                     0  20961  20961
True                   2865   7478  10343
All                    2865  28439  31304

Differing but existing codes: 7478
  same municipality: 85.5%
  same wijk:         52.6%

Municipalities with the most differences:
municipality
AMSTERDAM              1837
ALMERE                  527
SÚDWEST-FRYSLÂN         267
DIJK EN WAARD           231
HAARLEMMERMEER          220
HOLLANDS KROON          192
APELDOORN               166
ALPHEN AAN DEN RIJN     164
NOARDEAST-FRYSLÂN       164
LEEUWARDEN              151


In [7]:
j["summer_peak"] = j["peak_date"].dt.month.between(4, 9)

# 1. save the transformer-level file (without the geometry column)
keep = ["transformer_id", "buurtcode", "buurtnaam", "gemeentenaam", "buurtcode_reported", "method",
        "municipality", "postcode", "lat", "lon", "peak_kw", "peak_date", "capacity_kw", "nbl_limit_kw",
        "utilisation", "utilisation_capped", "above_nbl", "above_capacity", "summer_peak"]
j[keep].to_csv(PROCESSED / "transformers_buurt2023.csv", index=False)

# 2. aggregate per buurt
bs = j.groupby("buurtcode").agg(
    n_transformers=("transformer_id", "size"),
    capacity_kw_total=("capacity_kw", "sum"),
    peak_kw_total=("peak_kw", "sum"),
    util_max=("utilisation_capped", "max"),
    n_above_nbl=("above_nbl", "sum"),
    n_above_capacity=("above_capacity", "sum"),
    share_summer_peaks=("summer_peak", "mean"),
).reset_index()

bs["util_weighted"] = (bs["peak_kw_total"] / bs["capacity_kw_total"]).clip(upper=2.0)
bs["stress_nbl"] = (bs["n_above_nbl"] > 0).astype(int)            # at least 1 transformer above NBL
bs["stress_capacity"] = (bs["n_above_capacity"] > 0).astype(int)  # at least 1 transformer above capacity

bs.to_csv(PROCESSED / "buurt_transformer_stress.csv", index=False)

print("Buurten with transformers:", len(bs))
print("\nTransformers per buurt:")
print(bs["n_transformers"].describe().round(1).to_string())
print(f"\nBuurten with stress_nbl = 1:      {bs['stress_nbl'].sum():,} ({100 * bs['stress_nbl'].mean():.1f}%)")
print(f"Buurten with stress_capacity = 1: {bs['stress_capacity'].sum():,} ({100 * bs['stress_capacity'].mean():.1f}%)")
print(f"Average share of summer peaks:    {100 * bs['share_summer_peaks'].mean():.1f}%")
print("\nSaved: transformers_buurt2023.csv and buurt_transformer_stress.csv")

Buurten with transformers: 5523

Transformers per buurt:
count    5523.0
mean        5.7
std         5.0
min         1.0
25%         2.0
50%         4.0
75%         7.0
max        70.0

Buurten with stress_nbl = 1:      560 (10.1%)
Buurten with stress_capacity = 1: 364 (6.6%)
Average share of summer peaks:    98.6%

Saved: transformers_buurt2023.csv and buurt_transformer_stress.csv
